<a href="https://colab.research.google.com/github/Innovatewithapple/TransformersProjects/blob/main/BBCNewsGPTDecoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
gpreda_bbc_news_rss_feeds_path = kagglehub.notebook_output_download('gpreda/bbc-news-rss-feeds')

print('Data source import complete.')


In [ ]:
!pip install transformers

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader,Dataset
from transformers import AutoModelForCausalLM,AutoModel,AutoTokenizer
import os
from google.colab import userdata
from sklearn.model_selection import train_test_split
import pandas as pd
from tqdm import tqdm
import gc

In [ ]:
# os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
# os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

In [ ]:
# !kaggle datasets download -d alfathterry/bbc-full-text-document-classification

In [ ]:
# !unzip -q bbc-news.zip -d ./datafolder/

In [ ]:
df = pd.read_csv('/kaggle/input/notebooks/gpreda/bbc-news-rss-feeds/bbc_news.csv')

In [ ]:
df.isnull().sum()

title          0
pubDate        0
guid           0
link           0
description    0
dtype: int64

In [ ]:
# df = df.dropna(subset=['statement'])

In [ ]:
# df = df[df['statement'].str.strip() != ""]

In [ ]:
autoToken = AutoTokenizer.from_pretrained('gpt2')
# 2. THE CRITICAL FIX: GPT-2 needs a padding token
# We tell it to use the 'End of String' token as padding
autoToken.pad_token = autoToken.eos_token

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)
print("GPU Count:", torch.cuda.device_count())

Device: cuda
GPU Count: 2


In [ ]:
max_len = 256
vocab_size = autoToken.vocab_size

In [ ]:
statement_text = df['description'].astype(str).values

In [ ]:
x_train,x_test = train_test_split(statement_text,test_size=0.3,random_state=42)

In [ ]:
train_data = autoToken(text=list(x_train),padding='max_length',max_length=max_len+1,add_special_tokens=True,truncation=True,return_attention_mask=True,return_tensors='pt')
val_data = autoToken(text=list(x_test),padding='max_length',max_length=max_len+1,add_special_tokens=True,truncation=True,return_attention_mask=True,return_tensors='pt')

In [ ]:
class SentimentDataset(Dataset):
  def __init__(self,encoding):
    self.encoding = encoding

  def __len__(self):
    return len(self.encoding['input_ids'])

  def __getitem__(self,idx):
    ids = self.encoding['input_ids'][idx]
    mask = self.encoding['attention_mask'][idx]

    return {
        'input_ids':ids[:-1],
        'target_ids':ids[1:],
        'attention_mask':mask[:-1]
    }

In [ ]:
train_d = SentimentDataset(encoding=train_data)
val_d = SentimentDataset(encoding=val_data)

In [ ]:
train_ds = DataLoader(dataset=train_d,batch_size=16,shuffle=True,pin_memory=True,num_workers=2)
val_ds = DataLoader(dataset=val_d,batch_size=16,shuffle=False,pin_memory=True,num_workers=2)

GPT-PreTrained

In [ ]:
class CustomGPT2Writer(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # 1. The Pre-trained 'Writer' Brain (768 features)
        self.gpt2 = AutoModel.from_pretrained('gpt2')

        # 2. YOUR OWN CUSTOM LAYERS (Just like you wanted!)
        self.dropout = nn.Dropout(0.2)
        # For a writer, the output MUST be the vocab_size
        self.output_layer = nn.Linear(768, vocab_size)

    def forward(self, input_ids, attention_mask):
        # 1. Get features from the brain
        outputs = self.gpt2(input_ids=input_ids, attention_mask=attention_mask)

        # 2. Grab the hidden states (The words' meaning)
        # Shape: [Batch, Seq_Len, 768]
        x = outputs.last_hidden_state

        # 3. Apply your own custom logic
        x = self.dropout(x)

        # 4. Pass through your own custom output layer
        # Result: [Batch, Seq_Len, Vocab_Size]
        logits = self.output_layer(x)

        return logits

In [ ]:
# Use your custom class that outputs raw logits [prev]
model = CustomGPT2Writer(vocab_size=autoToken.vocab_size)

# Wrap it in DataParallel for your multi-GPU setups
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
model = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
criterion = nn.CrossEntropyLoss()
scaler = torch.cuda.amp.GradScaler()

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2Model LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_57/2507378013.py:11: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [ ]:
epochs = 4

# ==================== 2. MAIN CORRECTIONS LOOP ====================
for epoch in range(epochs):

    # -------------------- TRAINING PHASE --------------------
    model.train()
    train_loss = 0

    progress_bar_train = tqdm(train_ds, desc=f"Epoch {epoch+1}")
    for batch in progress_bar_train:
        # Unpack, migrate to GPU, and drop hidden dimensions [prev]
        ids = batch['input_ids'].to(device).squeeze(1)
        mask = batch['attention_mask'].to(device).squeeze(1)
        targets = batch['target_ids'].to(device).squeeze(1)

        optimizer.zero_grad()

        # Enclose manual calculations inside low-memory 16-bit autocast [prev]
        with torch.cuda.amp.autocast():
            # Returns raw hidden logits [Batch, Seq_Len, Vocab_Size] [prev]
            logits = model(ids, mask)

            # Manual 3D -> 2D flattening outside parallelized blocks [prev]
            loss = criterion(logits.view(-1, logits.size(-1)), targets.view(-1))

        # Execute backward and step operations via the gradient scaler [prev]
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        progress_bar_train.set_postfix(loss=loss.item())

    # Calculate metrics for the whole training epoch [prev]
    avg_train_loss = train_loss / len(train_ds)
    train_perplexity = torch.exp(torch.tensor(avg_train_loss))
    print(f"\nTrain Loss: {avg_train_loss:.4f} | Train Perplexity: {train_perplexity:.2f}")

    # ==================== VALIDATION PHASE ====================
    model.eval()
    total_val_loss = 0

    progress_bar_val = tqdm(val_ds, desc='Validation')
    with torch.no_grad():
        for batch in progress_bar_val:
            v_ids = batch['input_ids'].to(device).squeeze(1)
            v_mask = batch['attention_mask'].to(device).squeeze(1)
            v_targets = batch['target_ids'].to(device).squeeze(1)

            with torch.cuda.amp.autocast():
                v_logits = model(v_ids, v_mask)
                v_loss = criterion(v_logits.view(-1, v_logits.size(-1)), v_targets.view(-1))

            # PRO TIP: Pulling .item() breaks the tensor connection
            # and prevents memory leakage into the next epoch
            total_val_loss += v_loss.item()
            progress_bar_val.set_postfix(loss=v_loss.item())

    # Calculate metrics outside the loop
    avg_val_loss = total_val_loss / len(val_ds)
    val_perplexity = torch.exp(torch.tensor(avg_val_loss))
    print(f"Epoch {epoch+1} | Val Loss: {avg_val_loss:.4f} | Val Perplexity: {val_perplexity:.2f}")
    print("=" * 60)

    # 💥 THE CRITICAL MEMORY FLUSH ADDITION
    # This force-cleans the validation tensor cache lines before Epoch 2 starts!
    gc.collect()
    torch.cuda.empty_cache()

Epoch 1:   0%|          | 0/1843 [00:00<?, ?it/s]/tmp/ipykernel_57/249388049.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1: 100%|██████████| 1843/1843 [13:19<00:00,  2.30it/s, loss=0.422]



Train Loss: 0.7453 | Train Perplexity: 2.11


Validation:   0%|          | 0/790 [00:00<?, ?it/s]/tmp/ipykernel_57/249388049.py:51: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Validation: 100%|██████████| 790/790 [02:12<00:00,  5.98it/s, loss=0.448]


Epoch 1 | Val Loss: 0.4767 | Val Perplexity: 1.61


Epoch 2: 100%|██████████| 1843/1843 [13:20<00:00,  2.30it/s, loss=0.385]


...

Epoch 3 | Val Loss: 0.3681 | Val Perplexity: 1.45


Epoch 4: 100%|██████████| 1843/1843 [13:19<00:00,  2.31it/s, loss=0.358]



Train Loss: 0.3438 | Train Perplexity: 1.41


Validation: 100%|██████████| 790/790 [02:11<00:00,  5.99it/s, loss=0.318]


Epoch 4 | Val Loss: 0.3481 | Val Perplexity: 1.42


In [ ]:
def generate_news(model, tokenizer, prompt, max_new_tokens=50):
    model.eval()

    # 1. Turn your headline prompt into token numbers
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

    # Extract the true inner model if DataParallel is active
    actual_model = model.module if isinstance(model, nn.DataParallel) else model

    for _ in range(max_new_tokens):
        with torch.no_grad():
            with torch.cuda.amp.autocast():
                # 2. Get the vocabulary predictions matrix
                logits = actual_model(input_ids, attention_mask=None)

        # 3. Focus entirely on the predictions for the VERY LAST word slot
        next_token_logits = logits[:, -1, :]

        # 4. Select the highest scoring word (Classification winner!)
        next_token = torch.argmax(next_token_logits, dim=-1).unsqueeze(0)

        # 5. Concatenate the new word token onto the sequence string array
        input_ids = torch.cat([input_ids, next_token], dim=-1)

        # Stop typing if the model hits the End-of-Text boundary token
        if next_token.item() == tokenizer.eos_token_id:
            break

    # 6. Translate the raw numbers back into readable English text
    return tokenizer.decode(input_ids, skip_special_tokens=True)[0]

# 💥 RUN THESE EXAMPLES IN A NEW CELL TO SEE IT WRITE
print("--- TEST 1 ---")
print(generate_news(model, autoToken, "The UK government announced"))

print("\n--- TEST 2 ---")
print(generate_news(model, autoToken, "A massive technological breakthrough"))


--- TEST 1 ---


/tmp/ipykernel_57/2148177564.py:12: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


The UK government announced a new law to allow asylum seekers to be held by the UK.

--- TEST 2 ---
A massive technological breakthrough has left the UK's largest city in a "serious condition".
